# Classificador de Qualidade de Lances de Xadrez

**Disciplina:** Paradigmas de Aprendizagem de Máquina  

---

## 1. Introdução

### Problema

Jogadores de xadrez de nível intermediário (rating 1200–1500 Chess.com, equivalente a 1400–1700 no Lichess) cometem erros frequentes durante o meio-jogo: peças penduradas, trocas desfavoráveis, descuido com a segurança do rei. Este projeto constrói um **classificador supervisionado** que distingue lances **bons** de **ruins** com base em features posicionais.

### Objetivo

Treinar e comparar dois modelos de classificação — **Decision Tree** (obrigatório, interpretável) e **Random Forest** (ensemble, para comparação) — demonstrando o pipeline completo de ML: coleta, rotulagem, engenharia de features, treino com tuning, avaliação e interpretação no domínio.

### Versões do pipeline

| Versão | Features | Descrição |
|--------|----------|-----------|
| **V1** | 33 posicionais | Material, mobilidade, segurança do rei, estrutura de peões, controle do centro, características do lance, contexto |
| **V2** | 52 (33 + 19 táticas) | V1 + peças indefesas, capturas com ganho, cravadas, rei avançado, tensão |

O notebook é controlado por duas flags na célula de configuração:
- **`VERSION`** (1 ou 2): seleciona qual conjunto de features, modelos e avaliação usar
- **`RERUN_PIPELINE`** (True/False): re-executa o pipeline do zero ou usa dados pré-computados

### Abordagem resumida

| Etapa | Método |
|-------|--------|
| Dados | Partidas reais do Lichess (CC0), filtradas por rating e tempo |
| Rotulagem | Avaliação posicional via Stockfish (depth 15): bom (≥ −50 cp), ruim (≤ −150 cp) |
| Features | V1: 33 posicionais · V2: 52 (posicionais + táticas) — `python-chess` |
| Modelos | Decision Tree + Random Forest, com `class_weight="balanced"` |
| Métricas | Foco em recall e F1 da classe "ruim" — detectar erros é o objetivo |

In [ ]:
import json
import os
import random
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.tree import export_text, plot_tree

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"font.size": 11, "figure.facecolor": "white"})
sns.set_style("whitegrid")

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = Path("data")

print(f"Diretório de trabalho: {os.getcwd()}")
print("Setup OK")

In [ ]:
VERSION = 2            # 1 = posicional (33 features), 2 = posicional + tática (52 features)
RERUN_PIPELINE = False  # True = re-executa todo o pipeline, False = usa dados pré-computados

_v_suffix = "" if VERSION == 1 else "_v2"
FEATURES_CSV = DATA_DIR / "features" / f"features{_v_suffix}.csv"
MODELS_DIR = DATA_DIR / "models" if VERSION == 1 else DATA_DIR / "models_v2"
EVAL_DIR = DATA_DIR / "evaluation" if VERSION == 1 else DATA_DIR / "evaluation_v2"

_ver_label = "V1 — posicional (33 features)" if VERSION == 1 else "V2 — posicional + tática (52 features)"
print(f"VERSION        = {VERSION}  ({_ver_label})")
print(f"RERUN_PIPELINE = {RERUN_PIPELINE}")
print(f"Features CSV   : {FEATURES_CSV}")
print(f"Modelos        : {MODELS_DIR}")
print(f"Avaliação      : {EVAL_DIR}")
print()
if RERUN_PIPELINE:
    print("⚠️  O pipeline completo será re-executado (pode levar >3 horas).")
else:
    print("Usando dados pré-computados.")

---

## 2. Coleta e Descrição dos Dados

### Fonte

Os dados vêm da **Lichess open database** ([database.lichess.org](https://database.lichess.org)), sob licença **CC0** (domínio público).

- **Ficheiro:** `lichess_db_standard_rated_2015-01.pgn.zst` (~272 MiB comprimido)
- **Formato:** PGN comprimido com Zstandard, processado em streaming (memória constante)

### Filtros aplicados

| Filtro | Critério | Justificativa |
|--------|----------|---------------|
| Rating | Ambos 1400–1700 (Lichess) | Faixa-alvo equivalente a 1200–1500 Chess.com |
| Tempo | Blitz/Rapid (3–10 min) | Jogos sérios mas com pressão temporal |
| Término | Normal (mate/resignação) | Evitar jogos decididos por timeout |
| Variante | Standard | Sem Chess960 etc. |
| Fase | Lances 8 a 40 | Meio-jogo (exclui abertura teórica e finais simplificados) |
| Amostragem | 10% das partidas válidas | Controle de volume, seed=42 |

In [ ]:
if RERUN_PIPELINE:
    from download_pgn import download

    PGN_URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2015-01.pgn.zst"
    PGN_PATH = DATA_DIR / "raw" / "lichess_db_standard_rated_2015-01.pgn.zst"

    if not PGN_PATH.exists():
        print("Baixando PGN do Lichess (~272 MiB)...")
        download(PGN_URL, PGN_PATH)
    else:
        print(f"PGN já existe: {PGN_PATH}")
else:
    print("⏭️  Download do PGN pulado (usando dados pré-computados).")

In [ ]:
if RERUN_PIPELINE:
    from filter_games import filter_and_sample, print_stats

    print("Filtrando e amostrando partidas...")
    print("Critérios: Elo 1400–1700, Blitz/Rapid, Normal, lances 8–40")
    print("Amostragem: 10%, seed=42, max 3000 partidas\n")

    stats = filter_and_sample(
        pgn_path=DATA_DIR / "raw" / "lichess_db_standard_rated_2015-01.pgn.zst",
        output_path=DATA_DIR / "filtered" / "moves_filtered.csv",
        sample_rate=0.10,
        seed=42,
        max_games=3000,
    )
    print_stats(stats)
else:
    print("⏭️  Filtragem pulada (usando CSV pré-computado).")

In [ ]:
df_filtered = pd.read_csv(DATA_DIR / "filtered" / "moves_filtered.csv")

n_games = df_filtered["game_site"].nunique()
n_moves = len(df_filtered)
avg_moves = n_moves / n_games

print(f"{'='*50}")
print(f"  DATASET FILTRADO")
print(f"{'='*50}")
print(f"  Partidas       : {n_games:,}")
print(f"  Lances (total) : {n_moves:,}")
print(f"  Lances/partida : {avg_moves:.1f}")
print(f"  Colunas        : {list(df_filtered.columns)}")
print()
df_filtered.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, title, color in [
    (axes[0], "white_elo", "Rating das Brancas", "#4C72B0"),
    (axes[1], "black_elo", "Rating das Pretas", "#DD8452"),
]:
    elos = df_filtered.groupby("game_site")[col].first()
    ax.hist(elos, bins=30, color=color, edgecolor="white", alpha=0.85)
    ax.set_xlabel("Elo")
    ax.set_ylabel("Partidas")
    ax.set_title(title)
    ax.axvline(elos.mean(), color="red", linestyle="--",
               label=f"Média: {elos.mean():.0f}")
    ax.legend()

plt.suptitle("Distribuição de Ratings no Dataset Filtrado", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 3. Rotulagem via Stockfish

### Conceito: centipawn loss (delta)

Para cada lance, a qualidade é medida pela **variação da avaliação** que ele causa:

$$\delta = \text{eval\_depois} - \text{eval\_antes}$$

onde ambas as avaliações são do ponto de vista do jogador que fez o lance. Um delta muito negativo significa que o lance **piorou** a posição.

### Limiares de classificação

| Classe | Condição | Significado | Analogia |
|--------|----------|-------------|----------|
| **Bom** | δ ≥ −50 cp | Perda ≤ 0.5 peão | Lance aceitável |
| **Descartado** | −150 < δ < −50 | Imprecisão intermediária | Zona cinzenta — ambíguo |
| **Ruim** | δ ≤ −150 cp | Perda ≥ 1.5 peão | Erro claro |

A **zona cinzenta** é descartada para reduzir ruído: lances que perdem entre 0.5 e 1.5 peão são ambíguos demais para rotular com confiança.

### Configuração do Stockfish

| Parâmetro | Valor | Justificativa |
|-----------|-------|---------------|
| Engine | Stockfish 18 | State-of-the-art, open source |
| Profundidade | 15 | Detecta táticas de 3–4 lances; equilíbrio custo/qualidade |
| Workers | 6 paralelos | Cada um com 1 thread + 64 MB hash |
| Caching | eval_after reutilizado como eval_before do lance seguinte | Corta avaliações quase pela metade |
| Tempo total | ~174 minutos | Apple M4 |

In [ ]:
if RERUN_PIPELINE:
    from label_moves import run as label_run

    print("Rotulando lances com Stockfish (depth 15, 6 workers)...")
    print("⚠️  Este passo leva ~174 minutos.\n")

    label_run(num_workers=6)
else:
    print("⏭️  Rotulagem pulada (usando CSVs pré-computados).")
    print(f"   → {DATA_DIR / 'labeled' / 'moves_all_scored.csv'}")
    print(f"   → {DATA_DIR / 'labeled' / 'moves_labeled.csv'}")

In [ ]:
df_scored = pd.read_csv(DATA_DIR / "labeled" / "moves_all_scored.csv")
df_labeled = pd.read_csv(DATA_DIR / "labeled" / "moves_labeled.csv")

n_bom = (df_labeled["label"] == "bom").sum()
n_ruim = (df_labeled["label"] == "ruim").sum()
n_desc = len(df_scored) - len(df_labeled)

print(f"{'='*50}")
print(f"  ROTULAGEM")
print(f"{'='*50}")
print(f"  Lances avaliados      : {len(df_scored):,}")
print(f"  Bom (δ ≥ −50 cp)      : {n_bom:,} ({n_bom/len(df_scored)*100:.1f}%)")
print(f"  Descartado (cinzenta) : {n_desc:,} ({n_desc/len(df_scored)*100:.1f}%)")
print(f"  Ruim (δ ≤ −150 cp)    : {n_ruim:,} ({n_ruim/len(df_scored)*100:.1f}%)")
print(f"  ──────────────────────")
print(f"  Dataset final         : {len(df_labeled):,} (bom + ruim)")
print(f"  Ratio bom:ruim        : {n_bom/n_ruim:.2f}:1")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
deltas = df_scored["delta_cp"].clip(-2000, 2000)
ax.hist(deltas, bins=80, color="#4C72B0", edgecolor="white", alpha=0.85)
ax.axvline(-50, color="green", linestyle="--", linewidth=1.5, label="Limiar bom (−50)")
ax.axvline(-150, color="red", linestyle="--", linewidth=1.5, label="Limiar ruim (−150)")
ax.set_xlabel("Delta (centipawns)")
ax.set_ylabel("Lances")
ax.set_title("Distribuição de Delta CP")
ax.legend(fontsize=9)

ax = axes[1]
counts = df_scored["label"].value_counts()
colors_map = {"bom": "#55A868", "ruim": "#C44E52", "descartado": "#8C8C8C"}
order = ["bom", "descartado", "ruim"]
bars = ax.bar(order, [counts.get(c, 0) for c in order],
              color=[colors_map.get(c, "gray") for c in order], edgecolor="white")
for bar, c in zip(bars, order):
    val = counts.get(c, 0)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f"{val:,}\n({val/len(df_scored)*100:.1f}%)", ha="center", fontsize=10)
ax.set_ylabel("Lances")
ax.set_title("Distribuição por Classe")

ax = axes[2]
bom_delta = df_labeled[df_labeled["label"] == "bom"]["delta_cp"].clip(-3000, 3000)
ruim_delta = df_labeled[df_labeled["label"] == "ruim"]["delta_cp"].clip(-3000, 3000)
bp = ax.boxplot([bom_delta, ruim_delta], labels=["Bom", "Ruim"],
                patch_artist=True, widths=0.5)
bp["boxes"][0].set_facecolor("#55A868")
bp["boxes"][1].set_facecolor("#C44E52")
ax.set_ylabel("Delta (centipawns)")
ax.set_title("Delta CP por Classe")

plt.suptitle("Análise da Rotulagem", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 4. Engenharia de Features

### Features V1 — Posicionais (33)

| Grupo | Qtd. | Features | Extração |
|-------|------|----------|----------|
| **Material** | 11 | Contagem por tipo/cor + diferença ponderada | `board.pieces()` |
| **Mobilidade** | 3 | Lances legais (jogador/adversário/diferença) | `board.legal_moves` |
| **Segurança do rei** | 4 | Roque, direito a roque, escudo de peões | Posição do rei + peões adjacentes |
| **Estrutura de peões** | 4 | Dobrados, isolados, passados | Análise de colunas |
| **Controle do centro** | 3 | Ataques e ocupação de d4/d5/e4/e5 | `board.is_attacked_by()` |
| **Características do lance** | 6 | Captura, xeque, promoção, peça, destino | `board.is_capture()` etc. |
| **Contexto** | 2 | Número do lance, cor | Metadados da partida |

### Features V2 — Táticas (+19, total 52)

| Grupo | Qtd. | Features | Extração |
|-------|------|----------|----------|
| **Peças indefesas** | 5 | Hanging pieces/value (jogador/adversário), min attacker vs piece | `board.attackers()` + `board.is_attacked_by()` |
| **Capturas com ganho** | 4 | Ameaças contra jogador/adversário, valor máximo de ameaça | Análise de atacantes vs valor da peça |
| **Cravadas** | 2 | Peças cravadas (jogador/adversário) | Detecção de pins via raios de ataque |
| **Rei avançado** | 4 | Atacantes do rei, colunas abertas, casas de fuga | Análise da zona do rei |
| **Tensão** | 4 | Ataques totais, casas disputadas, peças sem defesa | Contagem de ataques e casas contestadas |

**Decisões de design:**
- Nenhum one-hot de casas individuais (64 casas = esparsidade sem ganho para árvores)
- Todas numéricas/inteiras, compatíveis diretamente com scikit-learn
- Sem normalização necessária para modelos baseados em árvore

In [ ]:
if RERUN_PIPELINE:
    from extract_features import run as extract_run

    n_feat = 52 if VERSION == 2 else 33
    print(f"Extraindo {n_feat} features V{VERSION} de 109,290 lances (6 workers)...")
    extract_run(num_workers=6, batch_size=1000, v2=(VERSION == 2))
else:
    print(f"⏭️  Extração de features pulada (usando CSV pré-computado).")
    print(f"   → {FEATURES_CSV}")

In [ ]:
df_features = pd.read_csv(FEATURES_CSV)
feature_cols = [c for c in df_features.columns if c != "label"]

print(f"{'='*50}")
print(f"  FEATURES — V{VERSION}")
print(f"{'='*50}")
print(f"  Fonte       : {FEATURES_CSV}")
print(f"  Linhas      : {len(df_features):,}")
print(f"  Features    : {len(feature_cols)}")
print(f"  Valores nulos: {df_features[feature_cols].isnull().sum().sum()}")
print(f"\nDistribuição do label:")
print(df_features["label"].value_counts().to_string())
print(f"\nEstatísticas descritivas:")
df_features[feature_cols].describe().round(2)

In [ ]:
FEATURE_TRANSLATION = {
    # V1 — posicionais
    "white_pawns": "Peões brancos", "white_knights": "Cavalos brancos",
    "white_bishops": "Bispos brancos", "white_rooks": "Torres brancas",
    "white_queens": "Damas brancas", "black_pawns": "Peões pretos",
    "black_knights": "Cavalos pretos", "black_bishops": "Bispos pretos",
    "black_rooks": "Torres pretas", "black_queens": "Damas pretas",
    "material_diff": "Dif. material", "legal_moves_player": "Legais (jogador)",
    "legal_moves_opponent": "Legais (adversário)", "mobility_diff": "Dif. mobilidade",
    "player_castled": "Rocou", "opponent_castled": "Adv. rocou",
    "player_can_castle": "Pode rocar", "king_pawn_shield": "Escudo rei",
    "player_doubled_pawns": "Peões dobrados", "player_isolated_pawns": "Peões isolados",
    "player_passed_pawns": "Passados (jog.)", "opponent_passed_pawns": "Passados (adv.)",
    "player_center_control": "Centro (jog.)", "opponent_center_control": "Centro (adv.)",
    "player_center_occupation": "Ocupação centro", "is_capture": "É captura",
    "is_check": "Dá xeque", "is_promotion": "É promoção",
    "moved_piece": "Peça movida", "move_to_center": "Move p/ centro",
    "move_to_extended_center": "Move p/ centro exp.",
    "move_number": "Nº do lance", "is_white": "Joga de brancas",
    # V2 — táticas
    "hanging_pieces_player": "Indefesas (jog.)",
    "hanging_pieces_opponent": "Indefesas (adv.)",
    "hanging_value_player": "Val. indefeso (jog.)",
    "hanging_value_opponent": "Val. indefeso (adv.)",
    "min_attacker_vs_piece_player": "Menor atac. vs peça",
    "threats_against_player": "Ameaças (jog.)",
    "threats_against_opponent": "Ameaças (adv.)",
    "max_threat_value_player": "Maior ameaça (jog.)",
    "max_threat_value_opponent": "Maior ameaça (adv.)",
    "pinned_pieces_player": "Cravadas (jog.)",
    "pinned_pieces_opponent": "Cravadas (adv.)",
    "king_attackers_player": "Atacantes rei (jog.)",
    "king_attackers_opponent": "Atacantes rei (adv.)",
    "king_open_files_player": "Col. abertas rei",
    "king_escape_squares_player": "Fuga do rei",
    "total_attacks_player": "Ataques totais (jog.)",
    "total_attacks_opponent": "Ataques totais (adv.)",
    "contested_squares": "Casas disputadas",
    "undefended_pieces_player": "Sem defesa (jog.)",
}

corr = df_features[feature_cols].corr()
labels_pt = [FEATURE_TRANSLATION.get(c, c) for c in feature_cols]

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, cmap="RdBu_r", center=0,
    xticklabels=labels_pt, yticklabels=labels_pt,
    annot=False, square=True, linewidths=0.5,
    cbar_kws={"shrink": 0.7, "label": "Correlação de Pearson"}, ax=ax,
)
ax.set_title("Mapa de Correlação entre Features", fontsize=15, pad=15)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

high_corr = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        r = abs(corr.iloc[i, j])
        if r > 0.7:
            high_corr.append((feature_cols[i], feature_cols[j], corr.iloc[i, j]))

if high_corr:
    print("Pares com |correlação| > 0.7:")
    for a, b, r in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f"  {FEATURE_TRANSLATION.get(a, a):25s}  ↔  {FEATURE_TRANSLATION.get(b, b):25s}  r = {r:+.3f}")
else:
    print("Nenhum par de features com |correlação| > 0.7.")

---

## 5. Treino dos Modelos

### Split dos dados

O dataset foi dividido em **70/15/15** (treino/validação/teste) com **estratificação** para preservar a proporção bom/ruim em cada split. O conjunto de teste só é usado **uma vez**, na avaliação final.

### Desbalanceamento

A classe "ruim" representa ~15.6% do dataset (ratio **5.4:1**). Para mitigar o viés:
- `class_weight="balanced"` em ambos os modelos — ajusta o peso da loss proporcionalmente à frequência de cada classe

### Modelos

| Modelo | Por quê | Hiperparâmetros tunados |
|--------|---------|------------------------|
| **Decision Tree** | Obrigatório. Regras interpretáveis para o domínio. | `max_depth` ∈ {3,5,7,10,15,∅}, `min_samples_leaf` ∈ {1,5,10,20}, `criterion` ∈ {gini, entropy} |
| **Random Forest** | Ensemble de árvores — reduz variância. | `n_estimators` ∈ {50,100,200}, `max_depth` ∈ {5,10,15,∅}, `min_samples_leaf` ∈ {1,5,10} |

### Tuning

Ambos via `GridSearchCV` com **5 folds** e métrica **F1** da classe "ruim".

In [ ]:
X = df_features.drop(columns=["label"])
y = (df_features["label"] == "ruim").astype(int)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=RANDOM_SEED
)

print(f"{'='*50}")
print(f"  SPLIT DOS DADOS")
print(f"{'='*50}")
for name, Xs, ys in [("Treino", X_train, y_train), ("Validação", X_val, y_val), ("Teste", X_test, y_test)]:
    n = len(ys)
    n_ruim = int(ys.sum())
    print(f"  {name:10s}: {n:>6,} amostras  (bom={n - n_ruim:,}, ruim={n_ruim:,}, {n_ruim/n*100:.1f}% ruim)")

In [ ]:
if RERUN_PIPELINE:
    from train_models import run as train_run

    print(f"Treinando modelos V{VERSION} (GridSearchCV, 5 folds)...")
    print("⚠️  Pode levar alguns minutos.\n")
    train_run(v2=(VERSION == 2))
else:
    print(f"⏭️  Treino pulado (usando modelos pré-treinados de {MODELS_DIR}).")

with open(MODELS_DIR / "feature_names.json") as f:
    feature_names = json.load(f)

dt = joblib.load(MODELS_DIR / "decision_tree.joblib")
rf = joblib.load(MODELS_DIR / "random_forest.joblib")

print(f"\n{'='*55}")
print(f"  DECISION TREE V{VERSION} — Melhores hiperparâmetros")
print(f"{'='*55}")
print(f"  criterion        : {dt.criterion}")
print(f"  max_depth        : {dt.max_depth}")
print(f"  min_samples_leaf : {dt.min_samples_leaf}")
print(f"  class_weight     : {dt.class_weight}")

print(f"\n{'='*55}")
print(f"  RANDOM FOREST V{VERSION} — Melhores hiperparâmetros")
print(f"{'='*55}")
print(f"  n_estimators     : {rf.n_estimators}")
print(f"  max_depth        : {rf.max_depth}")
print(f"  min_samples_leaf : {rf.min_samples_leaf}")
print(f"  class_weight     : {rf.class_weight}")

---

## 6. Avaliação

A avaliação é feita no **conjunto de teste** (16,394 lances) — nunca usado no treino nem no tuning.

### Métrica principal: F1 da classe "ruim"

A classe "ruim" é a mais importante: o objetivo é **detectar erros**. Um modelo que classifica tudo como "bom" teria ~84% de accuracy mas seria inútil.

| Métrica | O que mede | Por que importa |
|---------|------------|------------------|
| **Accuracy** | Acertos totais | Visão geral (enviesada pelo desbalanceamento) |
| **Recall (ruim)** | % de erros reais detectados | Não deixar erros escaparem |
| **Precision (ruim)** | % de alarmes que são reais | Evitar alarmes falsos |
| **F1 (ruim)** | Equilíbrio precision/recall | Métrica-chave |
| **ROC-AUC** | Capacidade discriminativa geral | Independente do threshold |

In [ ]:
y_pred_dt = dt.predict(X_test)
y_pred_rf = rf.predict(X_test)

print("=" * 60)
print("  DECISION TREE — Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred_dt, target_names=["bom", "ruim"]))

print("=" * 60)
print("  RANDOM FOREST — Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred_rf, target_names=["bom", "ruim"]))

In [ ]:
results = []
for name, model, y_pred in [
    ("Decision Tree", dt, y_pred_dt),
    ("Random Forest", rf, y_pred_rf),
]:
    y_proba = model.predict_proba(X_test)[:, 1]
    results.append({
        "Modelo": name,
        "Accuracy": f"{accuracy_score(y_test, y_pred):.4f}",
        "F1 (bom)": f"{f1_score(y_test, y_pred, pos_label=0):.4f}",
        "F1 (ruim)": f"{f1_score(y_test, y_pred, pos_label=1):.4f}",
        "Recall (ruim)": f"{(y_pred[y_test == 1] == 1).mean():.4f}",
        "Precision (ruim)": f"{(y_test[y_pred == 1] == 1).mean():.4f}",
        "ROC-AUC": f"{roc_auc_score(y_test, y_proba):.4f}",
    })

df_results = pd.DataFrame(results)
print("Tabela comparativa — Conjunto de teste:\n")
df_results

### 6.1 Matrizes de Confusão

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, name in [
    (axes[0], dt, "Decision Tree"),
    (axes[1], rf, "Random Forest"),
]:
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        display_labels=["bom", "ruim"],
        cmap="Blues", ax=ax, values_format="d",
    )
    ax.set_title(f"Matriz de Confusão — {name}", fontsize=13)

plt.tight_layout()
plt.show()

### 6.2 Feature Importance

Quais features os modelos consideram mais importantes para distinguir lances bons de ruins?

In [ ]:
def plot_fi(model, feature_names, title, color, ax, top_n=15):
    imp = model.feature_importances_
    idx = np.argsort(imp)[::-1][:top_n]
    labels = [FEATURE_TRANSLATION.get(feature_names[i], feature_names[i]) for i in idx]
    values = imp[idx]
    bars = ax.barh(range(top_n), values, color=color)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("Importância (Gini)")
    ax.set_title(title, fontsize=13)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_fi(dt, feature_names, "Top 15 Features — Decision Tree", "#4C72B0", axes[0])
plot_fi(rf, feature_names, "Top 15 Features — Random Forest", "#DD8452", axes[1])
plt.tight_layout()
plt.show()

### 6.3 Curvas ROC e Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
RocCurveDisplay.from_estimator(dt, X_test, y_test, ax=ax, name="Decision Tree")
RocCurveDisplay.from_estimator(rf, X_test, y_test, ax=ax, name="Random Forest")
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Aleatório")
ax.set_title("Curva ROC — Classe 'ruim'", fontsize=13)
ax.legend()

ax = axes[1]
PrecisionRecallDisplay.from_estimator(dt, X_test, y_test, ax=ax, name="Decision Tree")
PrecisionRecallDisplay.from_estimator(rf, X_test, y_test, ax=ax, name="Random Forest")
prevalence = y_test.mean()
ax.axhline(y=prevalence, color="k", linestyle="--", alpha=0.5,
           label=f"Prevalência ({prevalence:.3f})")
ax.set_title("Curva Precision-Recall — Classe 'ruim'", fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

### 6.4 Comparação Geral de Modelos

In [ ]:
metrics_data = {}
for name, model in [("Decision Tree", dt), ("Random Forest", rf)]:
    yp = model.predict(X_test)
    yproba = model.predict_proba(X_test)[:, 1]
    metrics_data[name] = {
        "Accuracy": accuracy_score(y_test, yp),
        "F1 (bom)": f1_score(y_test, yp, pos_label=0),
        "F1 (ruim)": f1_score(y_test, yp, pos_label=1),
        "Recall (ruim)": (yp[y_test == 1] == 1).mean(),
        "ROC-AUC": roc_auc_score(y_test, yproba),
    }

metric_names = list(next(iter(metrics_data.values())).keys())
x = np.arange(len(metric_names))
width = 0.30

fig, ax = plt.subplots(figsize=(11, 5))
for i, (model_name, vals) in enumerate(metrics_data.items()):
    values = [vals[m] for m in metric_names]
    bars = ax.bar(x + i * width, values, width, label=model_name,
                  color=["#4C72B0", "#DD8452"][i])
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Score")
ax.set_title("Comparação de Modelos — Conjunto de Teste", fontsize=14)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 6.5 Curvas de Aprendizado

As curvas de aprendizado mostram como o F1-score evolui com o tamanho do conjunto de treino. Servem para responder: "precisamos de mais dados?" e "há overfitting?"

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train_sizes_frac = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]

for ax, model, name in [
    (axes[0], dt, "Decision Tree"),
    (axes[1], rf, "Random Forest"),
]:
    sizes, train_scores, val_scores = learning_curve(
        model, X_train, y_train, cv=5,
        train_sizes=train_sizes_frac,
        scoring="f1", n_jobs=-1,
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)

    ax.fill_between(sizes, train_mean - train_std, train_mean + train_std,
                    alpha=0.1, color="#4C72B0")
    ax.fill_between(sizes, val_mean - val_std, val_mean + val_std,
                    alpha=0.1, color="#DD8452")
    ax.plot(sizes, train_mean, "o-", color="#4C72B0", label="Treino")
    ax.plot(sizes, val_mean, "o-", color="#DD8452", label="Validação (CV)")
    ax.set_xlabel("Tamanho do treino")
    ax.set_ylabel("F1-score")
    ax.set_title(f"Curva de Aprendizado — {name}", fontsize=13)
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 7. Interpretação e Exemplos

### 7.1 Regras da Árvore de Decisão

A grande vantagem da Decision Tree é a **interpretabilidade**: as regras de decisão podem ser traduzidas diretamente para conceitos de xadrez.

In [ ]:
translated_names = [FEATURE_TRANSLATION.get(f, f) for f in feature_names]

fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    dt, feature_names=translated_names,
    class_names=["Bom", "Ruim"],
    filled=True, rounded=True,
    max_depth=3, fontsize=9, ax=ax,
    impurity=False, proportion=True,
)
ax.set_title("Árvore de Decisão (primeiros 3 níveis)", fontsize=15, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
_rules_file = EVAL_DIR / f"decision_tree_rules_chess{'_v2' if VERSION == 2 else ''}.txt"
if _rules_file.exists():
    rules_text = _rules_file.read_text()
else:
    rules_text = Path("data/evaluation/decision_tree_rules_chess.txt").read_text()

print(rules_text[:3000])
if len(rules_text) > 3000:
    print("\n... (regras continuam)")

### 7.2 Interpretação das regras

As regras revelam padrões consistentes com a teoria de xadrez:

| Regra aprendida | Interpretação no domínio |
|-----------------|-------------------------|
| **1ª decisão: é captura?** | Capturas e lances quietos têm padrões de erro distintos — capturas envolvem troca de material, lances quietos são mais posicionais. |
| **Mobilidade do adversário > 34** | Posição aberta com muitas opções para o adversário → mais fácil errar. |
| **Número do lance 10–15 e 24+** | Transições (abertura→meio-jogo e meio-jogo→final) são fases mais propensas a erros. |
| **Presença de damas** | Posições com damas são mais táticas, aumentando a probabilidade de erros. |
| **Diferença material ≤ -3** | Jogador com desvantagem material significativa erra mais frequentemente. |

### 7.3 Análise qualitativa de erros

Vamos examinar exemplos concretos onde o modelo erra, para entender as limitações.

In [ ]:
_fp_file = EVAL_DIR / f"error_analysis_fp{'_v2' if VERSION == 2 else ''}.csv"
_fn_file = EVAL_DIR / f"error_analysis_fn{'_v2' if VERSION == 2 else ''}.csv"
if not _fp_file.exists():
    _fp_file = DATA_DIR / "evaluation" / "error_analysis_fp.csv"
    _fn_file = DATA_DIR / "evaluation" / "error_analysis_fn.csv"
df_fp = pd.read_csv(_fp_file)
df_fn = pd.read_csv(_fn_file)

def show_examples(df, error_type, model_name, n=5):
    subset = df[df["model"] == model_name].head(n)
    tipo = "FALSOS POSITIVOS" if error_type == "FP" else "FALSOS NEGATIVOS"
    desc = ('modelo disse "ruim", lance é bom' if error_type == "FP"
            else 'modelo disse "bom", lance é ruim')

    print(f"\n{'='*65}")
    print(f"  {tipo} — {model_name}")
    print(f"  ({desc})")
    print(f"{'='*65}")

    for i, (_, row) in enumerate(subset.iterrows(), 1):
        print(f"\n  {error_type}-{i}: {row['move_san']} (lance #{int(row['move_number'])}, {row['color']})")
        print(f"    Delta: {row['delta_cp']} cp | Partida: {row['game_site']}")
        feats = str(row.get("top_features", "")).split("; ")
        translated_feats = [
            f"{FEATURE_TRANSLATION.get(f.split('=')[0], f.split('=')[0])}={f.split('=')[1]}"
            for f in feats if "=" in f
        ]
        print(f"    Features: {'; '.join(translated_feats)}")

_dt_name = f"Decision Tree{' V2' if VERSION == 2 else ''}"
_rf_name = f"Random Forest{' V2' if VERSION == 2 else ''}"
show_examples(df_fp, "FP", _dt_name)
show_examples(df_fn, "FN", _dt_name)
print()
show_examples(df_fp, "FP", _rf_name)
show_examples(df_fn, "FN", _rf_name)

### 7.4 Discussão dos erros

**Falsos positivos** (modelo diz "ruim", lance é bom):
- Tipicamente lances quietos em posições com alta mobilidade do adversário — a árvore associa esses contextos a erros, mas o lance é adequado.
- Posições no final do meio-jogo (lance 28+) onde a árvore tem viés por fase.

**Falsos negativos** (modelo diz "bom", lance é ruim):
- **Erros táticos sutis**: capturas que parecem normais pelas features posicionais mas perdem material por cravadas, raio-x ou contra-ataques.
- Posições com avaliação estável (mobilidade equilibrada) onde o erro é específico da sequência de lances.

**Limitação fundamental**: as 33 features posicionais capturam o *ambiente* da posição mas não as **táticas concretas**. Um lance pode ser ruim por perder um cavalo cravado, mas não temos feature para detectar cravadas.

---

## 8. Conclusões e Trabalhos Futuros

### Resultados principais

| Aspecto | Resultado |
|---------|-----------|  
| **Dataset** | 109,290 lances de meio-jogo, 3,000 partidas (1400–1700 Lichess) |
| **Melhor modelo** | Random Forest — accuracy 0.68, F1-ruim 0.35, AUC 0.68 |
| **Decision Tree** | Accuracy 0.62, F1-ruim 0.33, AUC 0.65 — mas recall-ruim superior (0.60 vs 0.55) |
| **Top features** | Nº do lance, mobilidade, se é captura, diferença material |
| **Interpretabilidade** | Regras da DT coerentes com teoria de xadrez |

### Por que o F1-ruim é ~0.35?

Três fatores:
1. **Desbalanceamento 5.4:1** — classe minoritária é inherentemente difícil
2. **Features posicionais simples** — não capturam táticas concretas (cravadas, raio-x, etc.)
3. **Natureza do problema** — prever erros humanos com features estáticas é fundamentalmente difícil; erros dependem da *sequência* de lances, não só da posição

### O que aprendemos

- Features posicionais simples têm poder preditivo **limitado mas real** — o modelo é melhor que o aleatório (AUC > 0.5)
- A **fase da partida** e a **mobilidade** são os preditores mais fortes, o que faz sentido: erros são mais comuns em transições e posições complexas
- A Decision Tree oferece **transparência** valiosa: as regras ajudam a entender *que tipo* de situação o modelo associa a erros
- O Random Forest melhora a performance ao custo de interpretrabilidade

### Limitações

- Dataset limitado a uma faixa de rating e período (jan/2015)
- Features não capturam ameaças táticas — principal fonte de erros nesta faixa
- Classificação binária (bom/ruim) é simplista
- Rótulos dependem do Stockfish (depth 15) — avaliações imprecisas introduzem ruído

### Trabalhos futuros

1. **Features táticas**: detectar cravadas, raio-x, peças indefesas — potencialmente o maior ganho
2. **Classificação multiclasse**: brilhante / bom / imprecisão / erro / blunder
3. **Modelos por fase**: separar abertura, meio-jogo e final
4. **Gradient boosting** (XGBoost/LightGBM) com features enriquecidas
5. **Aplicação prática**: módulo que sugere ao jogador revisar posições onde o modelo prevê erro